In [ ]:
from pathlib import Path

# Works when the notebook is launched from either the repo root or scripts/.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "shp" / "NEAFinal.shp").exists() else cwd.parent
SHP_PATH = PROJECT_ROOT / "shp" / "NEAFinal.shp"
OUT_DIR = PROJECT_ROOT / "Datasets" / "EVI_NIRv_0p1deg_NEA"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Shapefile:", SHP_PATH)
print("Output folder:", OUT_DIR)


In [ ]:
# ===================================================
# Annual EVI and NIRv, 2001-2024, clipped to NEAFinal
# EVI:  MODIS/061/MOD13A2, 16-day Terra VI, 1 km
# NIRv: MODIS/061/MCD43A4, daily NBAR red/NIR, 500 m
# Output: NetCDF + GeoTIFF on a 0.1-degree grid
# Uses the same xee/xarray pattern as the CMIP6 daily notebook.
# ===================================================

import gc
import math
import traceback
from pathlib import Path

import ee
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import xarray as xr
import xee
from rasterio.transform import Affine
from shapely.geometry import mapping

START_YEAR = 2001
END_YEAR = 2024
YEARS = list(range(START_YEAR, END_YEAR + 1))

EXPORT_CRS = "EPSG:4326"
GRID_DEG = 0.1
EXPORT_SCALE_M = 11132  # approximately 0.1 degree at the equator
WRITE_YEARLY_NETCDF = False  # keep False for robust resume; GeoTIFFs are the main deliverable

cloud_project = "kaleem-55"  # your EE project id

try:
    ee.Initialize(
        project=cloud_project,
        opt_url="https://earthengine-highvolume.googleapis.com",
    )
except Exception:
    ee.Authenticate()
    ee.Initialize(
        project=cloud_project,
        opt_url="https://earthengine-highvolume.googleapis.com",
    )

print("Required packages loaded: ee, geopandas, numpy, pandas, rasterio, xarray, xee")
print("Earth Engine initialized for project:", cloud_project)

MOD13_EVI_PROJECTION = (
    ee.ImageCollection("MODIS/061/MOD13A2")
    .first()
    .select("EVI")
    .projection()
)
MCD43_NIR_PROJECTION = (
    ee.ImageCollection("MODIS/061/MCD43A4")
    .first()
    .select("Nadir_Reflectance_Band2")
    .projection()
)

print("Native EVI projection:", MOD13_EVI_PROJECTION.getInfo())
print("Native NIRv source projection:", MCD43_NIR_PROJECTION.getInfo())


In [ ]:
if not SHP_PATH.exists():
    raise FileNotFoundError(f"Shapefile not found: {SHP_PATH}")

gdf = gpd.read_file(SHP_PATH).to_crs("EPSG:4326")
gdf = gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty].copy()

# Dissolve to one geometry for clipping/export.
study_geom = gdf.geometry.union_all() if hasattr(gdf.geometry, "union_all") else gdf.geometry.unary_union
studyarea = ee.Geometry(mapping(study_geom), proj="EPSG:4326", geodesic=False)

# This installed xee backend does not support geometry/projection/scale kwargs.
# Use its supported fixed grid arguments instead: crs + crs_transform + shape_2d.
minx, miny, maxx, maxy = study_geom.bounds
download_minx = math.floor(minx / GRID_DEG) * GRID_DEG
download_miny = math.floor(miny / GRID_DEG) * GRID_DEG
download_maxx = math.ceil(maxx / GRID_DEG) * GRID_DEG
download_maxy = math.ceil(maxy / GRID_DEG) * GRID_DEG
DOWNLOAD_WIDTH = int(round((download_maxx - download_minx) / GRID_DEG))
DOWNLOAD_HEIGHT = int(round((download_maxy - download_miny) / GRID_DEG))
XEE_CRS = EXPORT_CRS
XEE_CRS_TRANSFORM = (GRID_DEG, 0, download_minx, 0, -GRID_DEG, download_maxy)
XEE_SHAPE_2D = (DOWNLOAD_WIDTH, DOWNLOAD_HEIGHT)  # xee expects (width, height)

# Rasterio wants Affine(a, b, c, d, e, f), i.e.:
# x = a * col + b * row + c; y = d * col + e * row + f.
# Keep this explicit instead of splatting the xee tuple.
RASTER_TRANSFORM = Affine(GRID_DEG, 0, download_minx, 0, -GRID_DEG, download_maxy)

print("Features:", len(gdf))
print("Geometry type:", study_geom.geom_type)
print("Bounding box:", study_geom.bounds)
print("Xee bounds:", (download_minx, download_miny, download_maxx, download_maxy))
print("Xee crs_transform:", XEE_CRS_TRANSFORM)
print("Xee shape_2d:", XEE_SHAPE_2D)


In [ ]:
def mask_mod13_evi(image):
    """Scale MOD13A2 EVI and keep good/marginal SummaryQA pixels."""
    evi_raw = image.select("EVI")
    qa = image.select("SummaryQA")
    valid_range = evi_raw.gte(-2000).And(evi_raw.lte(10000))
    quality = qa.lte(1)  # 0 good, 1 marginal; mask snow/ice and cloudy
    return (
        evi_raw.multiply(0.0001)
        .updateMask(valid_range.And(quality))
        .rename("EVI")
        .copyProperties(image, ["system:time_start"])
    )


def mask_mcd43_nirv(image):
    """Scale MCD43A4 red/NIR NBAR, QA-mask, and calculate NIRv = NIR * NDVI."""
    red_raw = image.select("Nadir_Reflectance_Band1")
    nir_raw = image.select("Nadir_Reflectance_Band2")
    qa_red = image.select("BRDF_Albedo_Band_Mandatory_Quality_Band1")
    qa_nir = image.select("BRDF_Albedo_Band_Mandatory_Quality_Band2")

    quality = qa_red.lte(1).And(qa_nir.lte(1))
    valid = red_raw.gte(0).And(red_raw.lte(32766)).And(nir_raw.gte(0)).And(nir_raw.lte(32766))

    red = red_raw.multiply(0.0001)
    nir = nir_raw.multiply(0.0001)
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    nirv = nir.multiply(ndvi).rename("NIRv")

    return nirv.updateMask(valid.And(quality)).copyProperties(image, ["system:time_start"])


def annual_evi(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")
    return (
        ee.ImageCollection("MODIS/061/MOD13A2")
        .filterDate(start, end)
        .filterBounds(studyarea)
        .map(mask_mod13_evi)
        .mean()
        .setDefaultProjection(MOD13_EVI_PROJECTION)
        .clip(studyarea)
        .rename("EVI")
        .set("year", year)
        .set("system:time_start", start.millis())
    )


def annual_nirv(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")
    return (
        ee.ImageCollection("MODIS/061/MCD43A4")
        .filterDate(start, end)
        .filterBounds(studyarea)
        .map(mask_mcd43_nirv)
        .mean()
        .setDefaultProjection(MCD43_NIR_PROJECTION)
        .clip(studyarea)
        .rename("NIRv")
        .set("year", year)
        .set("system:time_start", start.millis())
    )


def to_0p1deg(image):
    """Prepare a clipped float image; xee applies the coarser export scale."""
    return image.clip(studyarea).toFloat()


In [ ]:
print("Prepared annual Earth Engine builders:")
print("  Years:", START_YEAR, "to", END_YEAR, "count", len(YEARS))
print("  Each year will be opened separately with xee to avoid Earth Engine memory limits.")


In [ ]:
def open_single_year_image(image, variable):
    """Open one annual EE image through xee on the fixed 0.1-degree grid."""
    collection = ee.ImageCollection.fromImages([image])
    ds = xr.open_dataset(
        collection,
        engine="ee",
        crs=XEE_CRS,
        crs_transform=XEE_CRS_TRANSFORM,
        shape_2d=XEE_SHAPE_2D,
        n_images=1,
    )

    if variable not in ds.data_vars:
        first_var = list(ds.data_vars)[0]
        ds = ds.rename({first_var: variable})

    return ds.sortby("time") if "time" in ds.coords else ds


def valid_tif(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        with rasterio.open(path) as src:
            return src.count >= 1 and src.width > 0 and src.height > 0
    except Exception:
        return False


def get_spatial_dims(data_array):
    y_candidates = ["lat", "latitude", "y"]
    x_candidates = ["lon", "longitude", "x"]
    y_dim = next((name for name in y_candidates if name in data_array.dims), None)
    x_dim = next((name for name in x_candidates if name in data_array.dims), None)
    if y_dim is None or x_dim is None:
        raise RuntimeError(f"Could not identify spatial dimensions for {data_array.dims}")
    return y_dim, x_dim


def annual_data_and_profile(ds, variable, count=1):
    da = ds[variable]
    y_dim, x_dim = get_spatial_dims(da)

    if y_dim in da.coords and len(da[y_dim]) > 1 and float(da[y_dim][0]) < float(da[y_dim][-1]):
        da = da.sortby(y_dim, ascending=False)
    if x_dim in da.coords and len(da[x_dim]) > 1 and float(da[x_dim][0]) > float(da[x_dim][-1]):
        da = da.sortby(x_dim)

    if "time" in da.dims:
        da = da.isel(time=0)

    # This is the only Earth Engine pixel fetch for a yearly export.
    data = da.transpose(y_dim, x_dim).values.astype("float32")
    profile = {
        "driver": "GTiff",
        "height": data.shape[0],
        "width": data.shape[1],
        "count": count,
        "dtype": "float32",
        "crs": XEE_CRS,
        "transform": RASTER_TRANSFORM,
        "compress": "lzw",
        "nodata": np.nan,
    }
    return data, profile


def export_one_year(year, variable, image_builder, yearly_dir, nc_dir):
    yearly_dir = Path(yearly_dir)
    nc_dir = Path(nc_dir)
    yearly_dir.mkdir(parents=True, exist_ok=True)
    nc_dir.mkdir(parents=True, exist_ok=True)

    tif_path = yearly_dir / f"{variable}_{year}_NEA_0p1deg.tif"
    nc_path = nc_dir / f"{variable}_{year}_NEA_0p1deg.nc"
    tmp_tif_path = tif_path.with_suffix(".tmp.tif")

    if valid_tif(tif_path):
        print(f"[SKIP] {tif_path.name}")
        return tif_path

    if tif_path.exists():
        print(f"[WARN] Removing incomplete file: {tif_path.name}")
        tif_path.unlink()
    if tmp_tif_path.exists():
        tmp_tif_path.unlink()

    print(f"[PROC] {variable} {year}")
    ds = None
    try:
        image = to_0p1deg(image_builder(year))
        ds = open_single_year_image(image, variable)
        ds[variable].attrs.update({
            "units": "unitless",
            "year": year,
            "resolution_m": EXPORT_SCALE_M,
            "note": "QA-masked annual mean, clipped to NEAFinal, exported on 0.1-degree grid.",
        })

        data, profile = annual_data_and_profile(ds, variable, count=1)
        with rasterio.open(tmp_tif_path, "w", **profile) as dst:
            dst.write(data, 1)
            dst.set_band_description(1, f"{variable}_{year}")
        tmp_tif_path.replace(tif_path)

        if WRITE_YEARLY_NETCDF:
            ds.to_netcdf(nc_path, encoding={variable: {"zlib": True, "complevel": 4}})

    finally:
        if ds is not None:
            ds.close()
        gc.collect()

    print("  [OK]", tif_path.name)
    return tif_path


def stack_yearly_geotiffs(yearly_paths, stack_path, variable):
    missing = [path for path in yearly_paths if not valid_tif(path)]
    if missing:
        missing_names = ", ".join(path.name for path in missing)
        raise RuntimeError(f"Cannot stack {variable}; missing/incomplete yearly files: {missing_names}")

    stack_path = Path(stack_path)
    stack_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(yearly_paths[0]) as first:
        profile = first.profile.copy()
    profile.update(count=len(yearly_paths), dtype="float32", compress="lzw", nodata=np.nan)

    tmp_stack = stack_path.with_suffix(".tmp.tif")
    if tmp_stack.exists():
        tmp_stack.unlink()

    with rasterio.open(tmp_stack, "w", **profile) as dst:
        for band_index, (year, path) in enumerate(zip(YEARS, yearly_paths), start=1):
            with rasterio.open(path) as src:
                dst.write(src.read(1).astype("float32"), band_index)
                dst.set_band_description(band_index, f"{variable}_{year}")
    tmp_stack.replace(stack_path)

    print("Saved stack:", stack_path)


evi_dir = OUT_DIR / "EVI_Annual_NEA_0p1deg"
nirv_dir = OUT_DIR / "NIRv_Annual_NEA_0p1deg"
nc_dir = OUT_DIR / "yearly_nc"
evi_stack_out = OUT_DIR / f"EVI_annual_{START_YEAR}_{END_YEAR}_NEA_0p1deg.tif"
nirv_stack_out = OUT_DIR / f"NIRv_annual_{START_YEAR}_{END_YEAR}_NEA_0p1deg.tif"

try:
    evi_yearly_paths = []
    nirv_yearly_paths = []

    for year in YEARS:
        evi_yearly_paths.append(export_one_year(year, "EVI", annual_evi, evi_dir, nc_dir / "EVI"))
        nirv_yearly_paths.append(export_one_year(year, "NIRv", annual_nirv, nirv_dir, nc_dir / "NIRv"))

    stack_yearly_geotiffs(evi_yearly_paths, evi_stack_out, "EVI")
    stack_yearly_geotiffs(nirv_yearly_paths, nirv_stack_out, "NIRv")

except Exception:
    traceback.print_exc(limit=3)
    raise


In [ ]:
# Final metadata check for the stacked GeoTIFFs.
for path in [evi_stack_out, nirv_stack_out]:
    with rasterio.open(path) as src:
        print(path.name)
        print("  CRS:", src.crs)
        print("  Resolution:", src.res)
        print("  Bands:", src.count)
        print("  Bounds:", src.bounds)
        print("  Band 1:", src.descriptions[0])
        print("  Band last:", src.descriptions[-1])
